# CSC integration straight from compressed frames, with pyFAI

This notebook shows how to take a **pyFAI** integrator, pull out its sparse
pixel-to-bin matrix, and hand that matrix to `bslz4_to_sparse` so that each
bitshuffle-LZ4 compressed frame of an HDF5 dataset is decompressed **and**
integrated in one pass, without ever building the dense image.
(See issue #17. The ImageD11 FAZIT transform is a separate notebook.)

Three cases, in increasing order of reduction:

1. `integrate1d`: intensity vs radius.
2. `integrate2d`: intensity on a (radius, azimuth) grid.
3. 2D integration with a *sum over ring ranges*: a handful of
   intensity-vs-azimuth curves, and the large 2D array is never made.

For each one the result is checked against pyFAI itself, including the
normalisation.

## 0. Setup

The packages needed are `pyFAI`, `h5py`, `hdf5plugin`, `scipy`, `numpy`, and a
build of this repository. A build in `../lib` (`python3 setup.py build
--build-lib=lib`) is put first on `sys.path`, as the test scripts do, so it wins
over any older installed copy.

`b.build_info()` reports what the loaded native extension was compiled from:
`git describe` of the source tree (a tag when built clean on a tagged commit,
`-dirty` plus the modified files otherwise), a sha256 of the compiled sources,
the compiler and the build time. `setup.py build` recompiles only when that
sha256 differs from the sources in the tree.

In [1]:
import os, sys, time, pathlib, mmap
lib = pathlib.Path('..').resolve() / 'lib'
if os.environ.get('BSLZ4_TO_SPARSE_PATH'):
    sys.path.insert(0, os.environ['BSLZ4_TO_SPARSE_PATH'])
elif lib.exists():
    sys.path.insert(0, str(lib))

import numpy as np
import scipy.sparse as sp
import h5py, hdf5plugin
import pyFAI
from pyFAI.calibrant import get_calibrant
from pyFAI.method_registry import IntegrationMethod
import bslz4_to_sparse as b

info = b.build_info()
print('bslz4_to_sparse', b.version, 'from', b.__file__)
print('  built from git %s, sources sha256 %s, %s' % (info['git'], info['src_sha256'][:12], info['built_utc']))
print('  with', info['compiler'], 'for', info['platform'])
if info['modified']:
    print('  modified, uncommitted files:', ', '.join(info['modified']))
print('backends', b.available_backends())
print('pyFAI', pyFAI.version, '| h5py', h5py.__version__, '| hdf5plugin', hdf5plugin.version)

bslz4_to_sparse 0.0.21a1 from /home/esrf/wright/git/bslz4_to_sparse_llm/lib/bslz4_to_sparse/__init__.py
  built from git v0.0.16-67-g917593f, sources sha256 55d7d7551da0, 2026-10-02T11:42:40Z
  with x86_64-conda-linux-gnu-cc (conda-forge gcc 13.3.0-2) 13.3.0 for linux_x86_64
backends ('kcb', 'scal', 'sse')
pyFAI 2025.3.0 | h5py 3.13.0 | hdf5plugin 5.0.0


## 1. A dummy geometry, loaded with `pyFAI.load()`

A `.poni` file is a small text file. This one is made up: an Eiger2 4M, 25 cm
from the sample, a slightly tilted detector, 0.285 Å (43.5 keV) X-rays.

In [2]:
poni = '''# Nota: units are meter and radian
poni_version: 2
Detector: Eiger2_4M
Detector_config: {}
Distance: 0.25
Poni1: 0.081
Poni2: 0.078
Rot1: 0.01
Rot2: 0.02
Rot3: 0.03
Wavelength: 2.85e-11
'''
pathlib.Path('dummy.poni').write_text(poni)
ai = pyFAI.load('dummy.poni')
print(ai)
npix = ai.detector.shape[0] * ai.detector.shape[1]
print('pixels', npix, '| masked (module gaps)', int(ai.detector.mask.sum()))

Detector Eiger2 4M	 PixelSize= 75µm, 75µm	 BottomRight (3)
Wavelength= 2.850000e-11 m
SampleDetDist= 2.500000e-01 m	PONI= 8.100000e-02, 7.800000e-02 m	rot1=0.010000  rot2=0.020000  rot3=0.030000 rad
DirectBeamDist= 250.063 mm	Center: x=1006.666, y=1146.679 pix	Tilt= 1.281° tiltPlanRotation= 116.562° 𝛌= 0.285Å
pixels 4471016 | masked (module gaps) 260328


## 2. Test data

Real data would be any `(nframes, ni, nj)` dataset written with the bitshuffle
filter (HDF5 id 32008), chunked as `(1, ni, nj)`. pyFAI's own test images
(`pyFAI.test.utilstest.UtilsTest.getimage`) are downloaded from the web and are
single frames, so they need network access and do not give a 100+ frame series.

Here the data are **synthetic**: Gaussian powder rings (peak ~4 counts, so ~9 % non-zero pixels and x14 compression, sparse like typical scattering) at the LaB6 ring
positions for this geometry, with an azimuthal `cos(2chi)` modulation (texture-like)
and Poisson noise, 200 independent frames, stored as
uint16 with `hdf5plugin.Bitshuffle(nelems=0, cname='lz4')`. The file is
regenerated if missing and is git-ignored.

In [3]:
NFRAMES = 200
fname = 'synthetic_rings.h5'
shape = ai.detector.shape
if not os.path.exists(fname):
    cal = get_calibrant('LaB6'); cal.wavelength = ai.wavelength
    qarr, chi = ai.qArray(), ai.chiArray()          # nm^-1, radians
    mean = np.full(shape, 0.001)                    # tiny background
    for k, qk in enumerate(cal.get_peaks('q_nm^-1')[:25]):
        amp = 4.0 / (1 + 0.3 * k)
        texture = 1 + 0.5 * np.cos(2 * chi + 0.4 * k)   # azimuthal modulation
        mean += amp * texture * np.exp(-0.5 * ((qarr - qk) / 0.15) ** 2)
    rng = np.random.default_rng(0)
    with h5py.File(fname, 'w') as h5f:
        ds = h5f.create_dataset('data', shape=(NFRAMES,) + shape, dtype=np.uint16,
                                chunks=(1,) + shape,
                                **hdf5plugin.Bitshuffle(nelems=0, cname='lz4'))
        for i in range(NFRAMES):
            frame = rng.poisson(mean).astype(np.uint16)
            frame[ai.detector.mask.astype(bool)] = 0
            ds[i] = frame

h5f = h5py.File(fname, 'r')
ds = h5f['data']
raw = npix * 2 * ds.shape[0]
comp = sum(ds.id.get_chunk_info(i).size for i in range(ds.shape[0]))
frm0 = ds[0]
print(ds.shape, ds.dtype, 'codec', b.detect_codec(ds))
print('raw %.0f MB, compressed %.1f MB (x%.0f)' % (raw/1e6, comp/1e6, raw/comp))
print('non-zero pixels in frame 0: %.1f %%' % (100 * np.count_nonzero(frm0) / npix))

(200, 2162, 2068) uint16 codec 2
raw 1788 MB, compressed 131.9 MB (x14)
non-zero pixels in frame 0: 9.0 %


## 3. The matrix, the normalisation, and what "equal to pyFAI" means

pyFAI's CSC engine holds a sparse matrix **M** of shape `(nbins, npix)`:
column *p* lists which bins pixel *p* contributes to, and with what weight
(`data`, `indices`, `indptr`, all in pixel order). For one image `img`:

    sum_signal        = M @ img
    sum_normalization = M @ norm_pixel
    intensity         = sum_signal / sum_normalization

`norm_pixel` is the per-pixel correction pyFAI builds from solid angle,
polarization, flat field and so on. **It does not depend on the image**, so
`sum_normalization` is the same for every frame. `chunk2sparseCSC*` returns
`M @ img` (called `powder`) for each frame; the two recipes below give
the matching denominator.

* **Matrix arguments** change `M`: `npt`, `unit`, `radial_range`,
  `azimuth_range`, the pixel-splitting method (`'bbox'`/`'no'`/`'full'` with
  `'csc'`), and the detector mask.
* **Normalisation-only arguments** leave `M` alone and only change the
  denominator: `correctSolidAngle`, `polarization_factor`, `flat`,
  `normalization_factor`.

The denominator is read from pyFAI itself by integrating an image of
ones with the same arguments and taking `sum_normalization`. There is no
separate formula to keep in sync.

Not supported: `dark` (it must be subtracted from the image before
integration, which defeats the purpose), a per-call `mask` or `dummy` (put
pixels to exclude into the detector mask so they are baked into `M`), and error
propagation (variance needs a squared-weights matrix, not built here).

In [4]:
METHOD = ('bbox', 'csc', 'cython')

# the mask our code wants: 1 = use this pixel
valid = np.ascontiguousarray((1 - ai.detector.mask).astype(np.uint8))
ones = np.ones(shape, np.float32)

def get_engine(ai, dim):
    m = IntegrationMethod.select_one_available(METHOD, dim=dim)
    return ai.engines[m].engine

def safe_divide(signal, norm):
    # pyFAI returns 0 in empty bins; so do we (no NaN / inf)
    return np.divide(signal, norm, out=np.zeros(np.shape(signal), np.float64), where=norm > 0)

def to_scipy(engine, nbins):
    # A scipy.sparse.csc_matrix of shape (nbins, npix), built on pyFAI's own arrays
    return sp.csc_matrix((engine.data, engine.indices, engine.indptr),
                         shape=(nbins, len(engine.indptr) - 1))

## 4. Case 1: `integrate1d`

In [5]:
NPT = 1000
kw = dict(unit='q_nm^-1', correctSolidAngle=True, polarization_factor=0.95)

ref = ai.integrate1d(frm0, NPT, method=METHOD, **kw)       # pyFAI, for checking
one = ai.integrate1d(ones, NPT, method=METHOD, **kw)       # gives the denominator
M1 = get_engine(ai, 1)
norm1 = one.sum_normalization
print('M:', M1.bins, 'bins x', len(M1.indptr) - 1, 'pixels, nnz', M1.indices.size)

c2s = b.chunk2sparseCSC(valid, M1, dtype=np.uint16)
npx, (vals, adr), powder = c2s(ds.id.read_direct_chunk((0, 0, 0))[1], 0)
I1 = safe_divide(powder, norm1)

print('sum_signal  max |diff| %.3g  (max value %.3g)' % (abs(powder - ref.sum_signal).max(), ref.sum_signal.max()))
print('intensity   max |diff| %.3g  (max value %.3g)' % (abs(I1 - ref.intensity).max(), ref.intensity.max()))
assert np.allclose(I1, ref.intensity, rtol=1e-4, atol=1e-3)

M: 1000 bins x 4471016 pixels, nnz 7684482
sum_signal  max |diff| 0  (max value 8.33e+03)
intensity   max |diff| 1.14e-07  (max value 3.82)


`sum_signal` should agree essentially exactly (the same float32 weights are summed), and the
intensity to float32 rounding.
The chunk passed in is the raw *compressed* bytes from
`read_direct_chunk`; `npx` is the number of non-zero pixels found and
`(vals, adr)` are those pixels as sparse values and flat indices, available
for free for other uses.

### Recipe check: the normalisation-only arguments

Each row changes only what goes into the ones-image call (and the pyFAI
reference); the same matrix `M1` and the same `powder` are reused.
`normalization_factor` needs no special handling: pyFAI already folds it into
`sum_normalization`.

In [6]:
cases = [
    dict(correctSolidAngle=False, polarization_factor=None),
    dict(correctSolidAngle=True,  polarization_factor=None),
    dict(correctSolidAngle=True,  polarization_factor=0.99),
    dict(correctSolidAngle=True,  polarization_factor=0.95, normalization_factor=2.5),
    dict(correctSolidAngle=True,  polarization_factor=0.95,
         flat=np.linspace(0.8, 1.2, npix, dtype=np.float32).reshape(shape)),
]
for extra in cases:
    k = dict(unit='q_nm^-1', **extra)
    r = ai.integrate1d(frm0, NPT, method=METHOD, **k)
    n = ai.integrate1d(ones, NPT, method=METHOD, **k)
    I = safe_divide(powder, n.sum_normalization)
    ok = np.allclose(I, r.intensity, rtol=1e-4, atol=1e-3)
    print('%-5s %s' % ('OK' if ok else 'FAIL', {a: (v if a != 'flat' else 'flat') for a, v in extra.items()}))

OK    {'correctSolidAngle': False, 'polarization_factor': None}


OK    {'correctSolidAngle': True, 'polarization_factor': None}


OK    {'correctSolidAngle': True, 'polarization_factor': 0.99}


OK    {'correctSolidAngle': True, 'polarization_factor': 0.95, 'normalization_factor': 2.5}


OK    {'correctSolidAngle': True, 'polarization_factor': 0.95, 'flat': 'flat'}


## 5. Case 2: `integrate2d`

For 2D, pyFAI's engine reports `bins` as a tuple `(npt_rad, npt_azim)`, which
`chunk2sparseCSC` does not take (it wants an integer `bins`). Wrapping the
same arrays in a `scipy.sparse.csc_matrix` of shape `(npt_rad*npt_azim, npix)`
works and copies nothing.

The flattened bin order is **radial-major**: flat index = `rad * npt_azim + azim`, so
`powder.reshape(npt_rad, npt_azim)`, and pyFAI's arrays (shape `(npt_azim,
npt_rad)`) are the transpose. The two sizes are deliberately different here so a
mix-up fails loudly.

In [7]:
NR, NA = 600, 90
kw2 = dict(unit='q_nm^-1', correctSolidAngle=True, polarization_factor=0.95)
ref2 = ai.integrate2d(frm0, NR, NA, method=METHOD, **kw2)
one2 = ai.integrate2d(ones, NR, NA, method=METHOD, **kw2)
M2 = to_scipy(get_engine(ai, 2), NR * NA)
print('M2', M2.shape, 'nnz', M2.nnz, '| %.0f MB' % ((M2.data.nbytes + M2.indices.nbytes) / 1e6))

c2s2 = b.chunk2sparseCSC(valid, M2, dtype=np.uint16)
_, _, powder2 = c2s2(ds.id.read_direct_chunk((0, 0, 0))[1], 0)

sig2 = powder2.reshape(NR, NA).T                      # -> (npt_azim, npt_rad) like pyFAI
I2 = safe_divide(sig2, one2.sum_normalization)
print('sum_signal  max |diff| %.3g  (max %.3g)' % (abs(sig2 - ref2.sum_signal).max(), ref2.sum_signal.max()))
print('intensity   max |diff| %.3g  (max %.3g)' % (abs(I2 - ref2.intensity).max(), ref2.intensity.max()))
print('empty bins: %d of %d' % ((one2.sum_normalization == 0).sum(), NR * NA))
assert np.allclose(I2, ref2.intensity, rtol=1e-4, atol=1e-2)

M2 (54000, 4471016) nnz 6484951 | 52 MB
sum_signal  max |diff| 0  (max 274)
intensity   max |diff| 2.37e-07  (max 6.05)
empty bins: 12884 of 54000


## 6. Case 3: sum over ring ranges, keep only intensity vs azimuth

Often the 2D map itself is not wanted, only the azimuthal dependence of a few
rings (texture work). Since everything is linear, the radial summation can be folded
into the matrix: if **W** is a `(nrings, npt_rad)` matrix with 1 where radial bin *r*
belongs to ring *k*, then

    S = kron(W, I_azim)                  # (nrings*npt_azim, npt_rad*npt_azim)
    G = S @ M2                           # (nrings*npt_azim, npix), still sparse

`G @ img` is the ring-summed signal directly, so a frame is decoded into
`nrings * npt_azim` numbers instead of a full 2D map. The rings can be any
width and need not be equally wide or contiguous.

**Normalisation:** the denominator is summed over the same bins, `S @ sum_normalization`, then
divided. This is the ratio of sums (a correct weighted merge) and **not** the mean
of the per-bin intensities.
The radial ranges are cut on the fine radial grid, so the ring edges are
quantised to the width of one radial bin.

In [8]:
q = ref2.radial                                        # bin centres, q in nm^-1
lab6 = get_calibrant('LaB6'); lab6.wavelength = ai.wavelength   # needed for get_peaks
q_peaks = np.array(lab6.get_peaks('q_nm^-1'))[:4]               # first 4 LaB6 rings
# arbitrary, unequal widths: +-(3%, 4%, 5%, 6%) around each ring
rings = [(qp * (1 - w), qp * (1 + w)) for qp, w in zip(q_peaks, (0.03, 0.04, 0.05, 0.06))]
W = np.array([(q >= lo) & (q < hi) for lo, hi in rings], dtype=np.float64)
print('radial bins per ring:', W.sum(1).astype(int))

S = sp.kron(sp.csr_matrix(W), sp.identity(NA), format='csr')
G = (S @ M2).tocsc()
# the C code wants the same dtypes pyFAI uses: float32 data, int32 indices
G = sp.csc_matrix((G.data.astype(np.float32), G.indices.astype(np.int32),
                   G.indptr.astype(np.int32)), shape=G.shape)
norm3 = S @ one2.sum_normalization.T.ravel()
print('G', G.shape, 'nnz', G.nnz, '| %.1f MB (M2 was %.0f MB)' %
      ((G.data.nbytes + G.indices.nbytes) / 1e6, (M2.data.nbytes + M2.indices.nbytes) / 1e6))

c2s3 = b.chunk2sparseCSC(valid, G, dtype=np.uint16)
_, _, p3 = c2s3(ds.id.read_direct_chunk((0, 0, 0))[1], 0)
I3 = safe_divide(p3, norm3).reshape(len(rings), NA)

# reference: the same ring sums, taken from pyFAI's own 2D arrays
I3_ref = np.array([safe_divide(W[k] @ ref2.sum_signal.T, W[k] @ one2.sum_normalization.T)
                   for k in range(len(rings))])
print('max |diff| vs pyFAI ring sums: %.3g  (max %.3g)' % (abs(I3 - I3_ref).max(), I3_ref.max()))
assert np.allclose(I3, I3_ref, rtol=1e-4, atol=1e-3)
print('result shape (ring, azimuth):', I3.shape)

radial bins per ring: [ 6 11 17 23]
G (360, 4471016) nnz 354280 | 2.8 MB (M2 was 52 MB)
max |diff| vs pyFAI ring sums: 9.03e-09  (max 2.46)
result shape (ring, azimuth): (4, 90)


## 7. Timing over 200 frames

Four ways to get from the file to a normalised result, all timed **including**
reading the data from HDF5 and the final division:

* **pyFAI:** `ds[i]` (HDF5 decompresses in C), then `integrate1d` with the CSC
  method. pyFAI's CSC integrator uses several cores (OpenMP).
* **1 frame at a time:** `read_direct_chunk` + `chunk2sparseCSC`. Single-threaded.
* **batched:** `chunk2sparseCSCmulti` on batches of frames. Single-threaded.
* **case 3** batched: the ring-sum matrix `G`, the useful result for texture work.

The second block of rows forces the dense or the sparse route in the decoder; see the
comment in the cell. On data that compresses less than the threshold (say x6 instead of x14) the default
picks the dense route, which is about twice as slow on this matrix. The threshold is
a placeholder, not yet calibrated.

Only the first repeat may include a cold file cache, so each is run three times and the best
is reported. The machine is shared, so treat the numbers as indicative.

In [9]:
def best(f, n=3):
    ts = []
    for _ in range(n):
        t0 = time.perf_counter(); out = f(); ts.append(time.perf_counter() - t0)
    return min(ts), out

nf = ds.shape[0]
print('frames', nf, '| cores visible to this process', len(os.sched_getaffinity(0)))

def run_pyfai():
    return np.array([ai.integrate1d(ds[i], NPT, method=METHOD, **kw).intensity for i in range(nf)])

def run_single():
    out = np.empty((nf, NPT))
    for i in range(nf):
        _, _, p = c2s(ds.id.read_direct_chunk((i, 0, 0))[1], 0)
        out[i] = safe_divide(p, norm1)
    return out

def h5py_chunks(frames):
    # the compressed bytes of each frame, read through HDF5
    return [ds.id.read_direct_chunk((i, 0, 0))[1] for i in frames]

def make_batched(csc, norm, batch, read_chunks=h5py_chunks):
    multi = b.chunk2sparseCSCmulti(valid, csc, dtype=np.uint16)
    def run():
        out = []
        for s in range(0, nf, batch):
            _, _, p = multi(read_chunks(range(s, min(s + batch, nf))), 0)
            out.append(safe_divide(p, norm))
        return np.concatenate(out)
    return run

t_ref, r_ref = best(run_pyfai)
t_one, r_one = best(run_single)
t_bat, r_bat = best(make_batched(M1, norm1, 25))
t_c3,  r_c3  = best(make_batched(G, norm3, 25))

# The decoder chooses per block between a 'sparse' route (compact the non-zero
# pixels, then look up only those in the matrix) and a 'dense' route (product over
# every pixel). The choice is the block's compression ratio against
# set_dense_sparse_threshold() (default 8): above it, sparse. Force each to see it.
saved = b.get_dense_sparse_threshold()
forced = {}
for label, th in (('dense', 1e9), ('sparse', 0.0)):
    b.set_dense_sparse_threshold(th)
    forced[label] = best(make_batched(M1, norm1, 25))[0]
b.set_dense_sparse_threshold(saved)

print('%-28s %7.2f s  %7.1f ms/frame  %6.1f fps' % ('pyFAI (h5py + integrate1d)', t_ref, 1e3*t_ref/nf, nf/t_ref))
print('%-28s %7.2f s  %7.1f ms/frame  %6.1f fps' % ('bslz4 CSC, 1 frame/call', t_one, 1e3*t_one/nf, nf/t_one))
print('%-28s %7.2f s  %7.1f ms/frame  %6.1f fps' % ('bslz4 CSC, batches of 25', t_bat, 1e3*t_bat/nf, nf/t_bat))
for label, t in forced.items():
    print('%-28s %7.2f s  %7.1f ms/frame  %6.1f fps' % ('  batches of 25, forced ' + label, t, 1e3*t/nf, nf/t))
print('%-28s %7.2f s  %7.1f ms/frame  %6.1f fps' % ('bslz4 CSC, case 3 (G), 25', t_c3, 1e3*t_c3/nf, nf/t_c3))

# every frame, not just frame 0, checked against pyFAI
print('all frames vs pyFAI, 1D: max |diff| %.3g / %.3g' % (abs(r_one - r_ref).max(), abs(r_ref).max()))
print('single vs batched agree:', np.allclose(r_one, r_bat, rtol=0, atol=1e-9))

frames 200 | cores visible to this process 96


pyFAI (h5py + integrate1d)     11.36 s     56.8 ms/frame    17.6 fps
bslz4 CSC, 1 frame/call         1.57 s      7.9 ms/frame   127.4 fps
bslz4 CSC, batches of 25        1.01 s      5.0 ms/frame   198.3 fps
  batches of 25, forced dense    2.13 s     10.6 ms/frame    93.9 fps
  batches of 25, forced sparse    0.99 s      4.9 ms/frame   202.6 fps
bslz4 CSC, case 3 (G), 25       0.75 s      3.7 ms/frame   267.3 fps
all frames vs pyFAI, 1D: max |diff| 1.19e-07 / 3.95
single vs batched agree: True


### Bypassing HDF5: chunk offsets, then `mmap` (the fast path) or `os.pread`

The decoder only needs the compressed bytes of each frame. HDF5 is needed once,
to find out where those bytes are: `b.harvest_chunk_offsets(ds)` returns
`{frame: (byte offset in the file, nbytes)}`, read from the chunk index in a
single pass (`chunk_iter`, or `get_chunk_info` with older h5py), and
`b.pack_offsets_lengths` turns that into two arrays in frame order. It refuses
a dataset that is not chunked `(1, ni, nj)`, or a chunk stored with the
bitshuffle filter skipped.

After that the frames are read with no HDF5 library call at all:

* **`multi.decode_offsets(mm, offsets, lengths, cut)`, the fast path:** map the
  whole file once with `mmap` and hand the decoder the map plus slices of the two
  offset arrays. A batch then costs one Python call: no per-frame `bytes`
  object, no copy (the decoder reads straight from the page cache), and every
  chunk is checked to lie inside the file before any is decoded. This is the
  one to use.
* **`os.pread(fd, nbytes, offset)`**: for when a mapping is not wanted (a
  network filesystem, a file still being written). One system call per frame
  into a new `bytes` object, which goes into the ordinary `multi(buffers, cut)`.
  It needs no seek, so one file descriptor can be shared by several threads.
  Not available on Windows.
* **`memoryview` slices of the map** passed to `multi(buffers, cut)` are also
  zero-copy. They are shown only to separate the cost of the per-frame objects
  from the cost of the copy.

The rows that pass buffers reuse `make_batched` from the cell above with the chunk
reader swapped. Each reader runs all three cases (1D, 2D, ring sums), and every result
is checked against the h5py run of the same case.

**Caveat: this file is read from memory.** After the first pass it sits in the page
cache, so the table shows only what the read path adds on top of the decode, mostly
the copy of the ~660 kB chunk that h5py and `pread` make and the zero-copy rows avoid.
That is a few per cent, and more for the cheaper decodes. Real data usually comes over
a network filesystem, where the read itself can cost as much as the decode or more, and
varies with the filesystem, its caching and readahead, and the load on it. The ranking
of the readers can be different there, so measure on the real storage before choosing.


In [10]:
frame_offsets = b.harvest_chunk_offsets(ds)       # {frame: (byte offset, nbytes)}, the only HDF5 use
offsets, lengths = b.pack_offsets_lengths(frame_offsets, range(nf))
print('frame 0: %d bytes at file offset %d' % (lengths[0], offsets[0]))

fh = open(fname, 'rb')
mm = mmap.mmap(fh.fileno(), 0, access=mmap.ACCESS_READ)

def make_offsets(csc, norm, batch):
    # the fast path: the whole file as one buffer, chunks addressed by offset
    multi = b.chunk2sparseCSCmulti(valid, csc, dtype=np.uint16)
    def run():
        out = []
        for s in range(0, nf, batch):
            _, _, p = multi.decode_offsets(mm, offsets[s:s + batch], lengths[s:s + batch], 0)
            out.append(safe_divide(p, norm))
        return np.concatenate(out)
    return run

fd = os.open(fname, os.O_RDONLY)
def pread_chunks(frames):
    return [os.pread(fd, int(lengths[i]), int(offsets[i])) for i in frames]

whole = memoryview(mm)
def mmap_chunks(frames):
    return [whole[offsets[i]:offsets[i] + lengths[i]] for i in frames]

cases = {'1D': (M1, norm1),                                    # case 1, 1000 bins
         '2D': (M2, one2.sum_normalization.T.ravel()),         # case 2, 600 x 90, radial-major
         'rings': (G, norm3)}                                  # case 3, 4 rings x 90
readers = {'h5py read_direct_chunk': lambda csc, norm: make_batched(csc, norm, 25),
           'mmap + decode_offsets': lambda csc, norm: make_offsets(csc, norm, 25),
           'os.pread': lambda csc, norm: make_batched(csc, norm, 25, pread_chunks),
           'mmap memoryview slices': lambda csc, norm: make_batched(csc, norm, 25, mmap_chunks)}
runs = {(r, c): make(*cases[c]) for r, make in readers.items() for c in cases}

# the differences are a few per cent, smaller than the drift of a shared machine
# between cells, so the repeats are interleaved and the best of each is kept
times = {k: [] for k in runs}
res = {}
for _ in range(5):
    for k, run in runs.items():
        t0 = time.perf_counter(); res[k] = run(); times[k].append(time.perf_counter() - t0)

print('ms/frame, batches of 25   ' + ''.join('%9s' % c for c in cases))
for r in readers:
    print('%-26s' % r + ''.join('%9.2f' % (1e3 * min(times[r, c]) / nf) for c in cases))
first = list(readers)[0]
print('identical to the h5py runs:',
      all(np.array_equal(res[r, c], res[first, c]) for r in readers for c in cases))
print('h5py 1D identical to the batched run above:', np.array_equal(res[first, '1D'], r_bat))

del whole        # release the exported buffer, or mm.close() raises BufferError
mm.close(); fh.close(); os.close(fd)

frame 0: 659488 bytes at file offset 4536


ms/frame, batches of 25          1D       2D    rings
h5py read_direct_chunk         4.95     6.09     3.67
mmap + decode_offsets          4.92     5.98     3.58
os.pread                       4.92     6.05     3.71
mmap memoryview slices         4.86     5.93     3.59
identical to the h5py runs: True
h5py 1D identical to the batched run above: True


## 8. Summary

* `M = engine.data/indices/indptr` from pyFAI's CSC engine, used as-is (1D), wrapped in
  `scipy.sparse.csc_matrix((data, indices, indptr), shape=(nbins, npix))` (2D), or
  multiplied by a binning matrix `S` (ring sums).
* `chunk2sparseCSC*` returns `M @ img` from the compressed bytes.
* The denominator is `sum_normalization` from pyFAI on an image of ones with the
  same arguments (the same `S @` for ring sums). Divide where it is positive.
* Arguments that change the pixel-to-bin map change `M`; correction arguments change only the
  denominator. `dark`, per-call `mask`/`dummy` and errors are not covered.
* `chunk2sparseCSC` currently needs an integer `bins`; pyFAI's 2D engine gives a tuple, hence
  the scipy wrapper.
* HDF5 is only needed to find the chunks (`harvest_chunk_offsets`). After that, `mmap` the file
  and call `chunk2sparseCSCmulti.decode_offsets(mm, offsets, lengths, cut)`: one call per
  batch, no copies, and every chunk bounds-checked against the file.

In [11]:
h5f.close()